In [1]:
import pandas as pd
import numpy as np

# Load datasets
orders = pd.read_csv('olist_orders_dataset.csv')
items = pd.read_csv('olist_order_items_dataset.csv')
reviews = pd.read_csv('olist_order_reviews_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
category_translation = pd.read_csv('product_category_name_translation.csv')

In [2]:
# --- 1. Clean Orders Dataset ---
date_cols = ['order_purchase_timestamp', 'order_approved_at', 
             'order_delivered_carrier_date', 'order_delivered_customer_date', 
             'order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

In [3]:
# New calculated fields
orders['order_month'] = orders['order_purchase_timestamp'].dt.month
orders['order_year'] = orders['order_purchase_timestamp'].dt.year

In [4]:
# Delivery durations in days
orders['delivery_days'] = (orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']).dt.days
orders['estimated_delivery_days'] = (orders['order_estimated_delivery_date'] - orders['order_purchase_timestamp']).dt.days

orders['is_delivered'] = orders['order_status'] == 'delivered'
orders['is_late'] = orders['order_delivered_customer_date'] > orders['order_estimated_delivery_date']

In [5]:
# --- 2. Clean Items Dataset ---
# New calculated fields
items['item_total_value'] = items['price'] + items['freight_value']
items['freight_ratio'] = items['freight_value'] / items['item_total_value']
# Handle any zero-division edge cases
items['freight_ratio'] = items['freight_ratio'].fillna(0)

In [6]:
# --- 3. Clean Reviews Dataset ---
# Group review scores into categories
conditions = [
    (reviews['review_score'] <= 2),
    (reviews['review_score'] == 3),
    (reviews['review_score'] >= 4)
]
choices = ['Negative', 'Neutral', 'Positive']
reviews['review_group'] = np.select(conditions, choices, default='Unknown')

In [7]:
# --- 4. Merge Products with English Translations ---
products = products.merge(category_translation, on='product_category_name', how='left')
# Fill missing translations or missing categories with 'unknown'
products['product_category_name_english'] = products['product_category_name_english'].fillna('unknown')

In [8]:
items

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,item_total_value,freight_ratio
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,72.19,0.184098
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,259.83,0.076704
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,216.87,0.082400
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,25.78,0.496121
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,218.04,0.083196
...,...,...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41,343.40,0.126412
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53,386.53,0.094508
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95,116.85,0.145058
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72,64.71,0.134755


In [9]:
# Save all cleaned tables
orders.to_csv('olist_orders_cleaned.csv', index=False)
items.to_csv('olist_order_items_cleaned.csv', index=False)
reviews.to_csv('olist_order_reviews_cleaned.csv', index=False)
products.to_csv('olist_products_cleaned.csv', index=False)

print("Cleaned data successfully created!")

Cleaned data successfully created!
